In [1]:
import pymongo
import pandas as pd
from sqlalchemy import create_engine
import numpy as np

# Conexión a MongoDB
client = pymongo.MongoClient("mongodb://10.0.1.253:27017/")  # URL de conexión a tu servidor MongoDB
db = client["dengue_bd"]  # Selecciona tu base de datos
collection = db["usuarios"]  # Selecciona tu colección

db1 = client["hospitales_db"]  # Selecciona tu base de datos
collection2 = db1["ESSALUD"]  # Selecciona tu colección

# Consulta todos los documentos en la colección
cursor = collection.find()

cursor1 = collection2.find()

# Convierte los documentos a una lista de diccionarios
documents = list(cursor)
documents2 = list(cursor1)

# Cierra la conexión a MongoDB
client.close()

# Convierte los ObjectId a cadenas de texto
for doc in documents:
    doc['_id'] = str(doc['_id'])

for doc in documents2:
    doc['_id'] = str(doc['_id'])

# Crea un DataFrame de Pandas con los documentos
df = pd.DataFrame(documents)
df2 = pd.DataFrame(documents2)

# Convertir las columnas con arrays a cadenas de texto (strings)
df['mensaje'] = df['mensaje'].apply(lambda x: ', '.join(map(str, x)))
df['datosPers'] = df['datosPers'].apply(lambda x: ', '.join(map(str, x)))
df['ecuBa'] = df['ecuBa'].apply(lambda x: ', '.join(map(str, x)))
# Convertir la columna 'ubiPer' a cadenas de texto (strings), manejando valores NaN
df['ubiPer'] = df['ubiPer'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, (list, np.ndarray)) else str(x))
# Dividir la columna 'ubiPer' en varias columnas
df[['latitud', 'longitud', 'cas1', 'cas2', 'cas3', 'cas4', 'cas5', 'cas6']] = df['ubiPer'].str.split(', ', expand=True)

DB_USER='postgres'
DB_PASSWORD='AKindOfMagic'
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST='10.0.1.228'
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()


In [2]:
df.to_sql(name=f'chatbot_dengue', con=connection2, if_exists='replace', index=False,chunksize=10000)
df2.to_sql(name=f'ipress_ubicaciones', con=connection2, if_exists='replace', index=False,chunksize=10000)

400